# 1. Tokens and embeddings

**Goal:** understand how text becomes numbers, and how numbers become vectors.

This is the first step of every language model:

```
text -> token IDs -> embedding vectors -> transformer
```

Run the cells in order. Nothing here trains; we only inspect shapes and values.

## Setup

In [1]:
from pathlib import Path
import sys

REPO = Path.cwd()
while not (REPO / "DeepSeekFlashV4-Mini").exists():
    if REPO == REPO.parent:
        raise RuntimeError("Run this notebook from inside deepseek_flash_lab")
    REPO = REPO.parent

sys.path.insert(0, str(REPO / "DeepSeekFlashV4-Mini"))
sys.path.insert(0, str(REPO))
import torch

torch.manual_seed(0)
print("repo root:", REPO.name)
print("torch:", torch.__version__)

repo root: deepseek_flash_lab
torch: 2.13.0+cu130


## 1. Build a tokenizer

A tokenizer is a lookup table between strings and integers.
This lab uses a **character-level** tokenizer so the vocabulary stays tiny.
Production models use BPE, but the idea is identical.

In [2]:
from tokenizer import CharTokenizer

text = "the cat sat on the mat."
tok = CharTokenizer.from_text(text)

ids = tok.encode(text)
print("text      :", repr(text))
print("vocab size:", tok.vocab_size)
print("ids       :", ids)
print("decoded   :", repr(tok.decode(ids)))

text      : 'the cat sat on the mat.'
vocab size: 11
ids       : [10, 5, 4, 0, 3, 2, 10, 0, 9, 2, 10, 0, 8, 7, 0, 10, 5, 4, 0, 6, 2, 10, 1]
decoded   : 'the cat sat on the mat.'


**Notice:** `len(ids) == len(text)`. One character = one token here.

In [3]:
# The vocabulary is just a dictionary.
items = list(tok.stoi.items())[:10]
print("char -> id:", items)
print("id -> char:", {i: c for c, i in items})

char -> id: [(' ', 0), ('.', 1), ('a', 2), ('c', 3), ('e', 4), ('h', 5), ('m', 6), ('n', 7), ('o', 8), ('s', 9)]
id -> char: {0: ' ', 1: '.', 2: 'a', 3: 'c', 4: 'e', 5: 'h', 6: 'm', 7: 'n', 8: 'o', 9: 's'}


## 2. Embeddings: ID -> vector

`nn.Embedding` stores a trainable matrix `E` of shape `[vocab_size, d_model]`.
Looking up a token is a **row lookup**; no multiplication happens.

In [4]:
from model.embedding import TokenEmbedding

d_model = 8
emb = TokenEmbedding(tok.vocab_size, d_model)

x = emb(torch.tensor([ids]))
print("token ids :", tuple(x.shape), "-> [B, T]")
print("embeddings:", tuple(x.shape), "-> [B, T, d_model]")
print("weight    :", tuple(emb.weight.shape), "-> [vocab, d_model]")

token ids : (1, 23, 8) -> [B, T]
embeddings: (1, 23, 8) -> [B, T, d_model]
weight    : (11, 8) -> [vocab, d_model]


In [5]:
# Prove it is a lookup: the first row equals weight[first_id].
first_id = ids[0]
same = torch.equal(x[0, 0], emb.weight[first_id])
print("x[0,0] == E[first_id]:", same)
print("first vector:", [round(v, 3) for v in x[0, 0].tolist()])

x[0,0] == E[first_id]: True
first vector: [-1.535, -0.413, 0.966, 1.625, -0.366, -1.302, 0.099, 0.442]


**Notice:** the same token ID always starts from the same vector. Training
changes `E`, so the vector changes over time, but the lookup rule does not.

## 3. Batches

Models process a batch `[B, T]` at once. The embedding layer handles any
leading dimensions and returns `[B, T, d_model]`.

> This tokenizer maps unknown characters to the space token. If you type a
> character that was not in the training text, it silently becomes a space.
> That is a useful beginner lesson: a tiny vocabulary has blind spots.

In [6]:
batch_text = ["cat", "sat"]
length = 3
ids_2d = torch.tensor([
    (tok.encode(t)[:length] + [0] * length)[:length] for t in batch_text
])
print("ids_2d:", ids_2d.tolist(), "shape", tuple(ids_2d.shape))
x2 = emb(ids_2d)
print("x2    :", tuple(x2.shape))

ids_2d: [[3, 2, 10], [9, 2, 10]] shape (2, 3)
x2    : (2, 3, 8)


## Recap

- Tokenizer: text -> integer IDs (a dictionary).
- Embedding: ID -> learned vector (a matrix row lookup).
- Shapes: `[B, T]` -> `[B, T, d_model]`.
- Parameters: `vocab_size * d_model`.

**Try it:** `docs/EXERCISES.md` exercise 1 asks you to change `d_model` and
predict the parameter count before running the cell.